# WP2 — Probabilistic membership and substructure

Consumes `wp1_gaia_narrow.parquet`; writes `wp2_members.parquet` and a provenance manifest. The development zero-point fallback and diagonal-error approximation are explicitly recorded and must be replaced before publication.

## Before Section 1 — Why establish controls first

This cell fixes paths, imports the clustering and mixture-model tools, and fixes a random seed so every Monte Carlo result is reproducible. The provisional -0.017 mas zero point is a global development value, not the final Gaia DR3 correction; it propagates into distances, subgroup separation, 3D positions, masses, and the later supernova ledger.


In [1]:
# Section 1 — Imports, paths, and controls
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from zero_point import zpt
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler

ROOT = Path.cwd()
if ROOT.name != 'gaia_snr_history_cygnus': ROOT = Path('/Users/vdk/science/gaia_snr_history_cygnus')
INPUT = ROOT/'data/processed/wp1_gaia_narrow.parquet'; OUT = ROOT/'data/processed'; PROV = ROOT/'provenance'
assert INPUT.exists(), INPUT
N_MC = 100; SEED = 20260722
rng = np.random.default_rng(SEED)

## Before Section 2 — Why apply the Gaia zero-point correction and quality flags

- RUWE<1.4 rejects many poor or non-single-star astrometric fits; 
- visibility_periods_used>=8 requires minimally stable time sampling; 
- a present BP/RP excess diagnostic allows photometric-quality inspection; and positive errors prevent invalid sampling. 

The correction is applied before membership because parallax determines the association distance and the one/two-population test. The raw value, source-dependent zero point, and corrected value are all retained. Gaia's correction depends on G magnitude, ecliptic latitude, colour proxy, and five-/six-parameter solution type; it is not the global -0.017 mas approximation. These are analysis flags, not deletions from WP1. They affect completeness, contamination, IMF normalization, and the predicted supernova count. The BP/RP condition currently checks only non-nullness, and RUWE 1.4 is a conventional starting point, so both require sensitivity tests.


In [2]:
# Section 2 — Load catalogue and retain explicit quality flags
df = pd.read_parquet(INPUT)
# Apply the source-dependent Lindegren et al. Gaia DR3/EDR3 recipe.
# The published lookup ranges are explicit: G=[6,21], nu_eff(5p)=[1.1,1.9],
# and pseudocolour(6p)=[1.24,1.72]. We flag boundary cases before clipping
# so the package does not silently hide extrapolation/clipping.
zpt.load_tables()
ecl_lat = SkyCoord(ra=df['ra'].to_numpy()*u.deg, dec=df['dec'].to_numpy()*u.deg).barycentrictrueecliptic.lat.deg
is_5p = df['astrometric_params_solved'].eq(31).to_numpy(); is_6p = df['astrometric_params_solved'].eq(95).to_numpy()
g_out = ~df['phot_g_mean_mag'].between(6,21).to_numpy()
nu_out = is_5p & ~df['nu_eff_used_in_astrometry'].between(1.1,1.9).fillna(False).to_numpy()
pc_out = is_6p & ~df['pseudocolour'].between(1.24,1.72).fillna(False).to_numpy()
df['zero_point_boundary_flag'] = g_out | nu_out | pc_out
g_for_zpt = df['phot_g_mean_mag'].clip(6+1e-6,21-1e-6).to_numpy()
# The recipe ignores nu_eff for 6p and pseudocolour for 5p; finite placeholders
# are supplied for those irrelevant fields because the vectorized function expects numbers.
nu_for_zpt = df['nu_eff_used_in_astrometry'].fillna(1.5).clip(1.1+1e-6,1.9-1e-6).to_numpy()
pc_for_zpt = df['pseudocolour'].fillna(1.5).clip(1.24+1e-6,1.72-1e-6).to_numpy()
df['parallax_zero_point'] = zpt.get_zpt(g_for_zpt, nu_for_zpt, pc_for_zpt, ecl_lat, df['astrometric_params_solved'].to_numpy(), _warnings=False)
df['zero_point_reliable'] = ~df['zero_point_boundary_flag']
df['parallax_raw'] = df['parallax']
df['parallax_corrected'] = df['parallax_raw'] - df['parallax_zero_point']
assert df['parallax_zero_point'].notna().all(), 'Zero-point recipe returned invalid values'
df['parallax_snr'] = df['parallax_corrected']/df['parallax_error']
df['quality_pass'] = (df['ruwe'].fillna(np.inf)<1.4) & (df['visibility_periods_used'].fillna(0)>=8) & df['phot_bp_rp_excess_factor'].notna() & df[['parallax_error','pmra_error','pmdec_error']].gt(0).all(axis=1) & df['zero_point_reliable']
analysis = df.loc[df.quality_pass].copy()
features = ['l_deg','b_deg','parallax_corrected','pmra','pmdec']
analysis = analysis.loc[analysis[features].notna().all(axis=1)].copy()
X = RobustScaler().fit_transform(analysis[features])
print('input',len(df),'analysis',len(analysis))

input 245843 analysis 232161


## Diagnostic cell — inspect, do not analyse

This diagnostic cell does not alter the catalogue. It reports the row count, a small preview, data types, missing-value fractions, and the number of sources passing the quality flag. These checks make the zero-point and quality-filter handoff visible without displaying the entire catalogue.


In [3]:
print(f'Rows: {len(df):,}; columns: {len(df.columns)}')
display(df[['source_id','parallax_raw','parallax_zero_point','parallax_corrected','parallax_error','quality_pass']].head())
display(df.dtypes.rename('dtype').to_frame().head(20))
display(df.isna().mean().sort_values(ascending=False).head(12).rename('null_fraction').to_frame())
print(f'Quality-passing rows: {int(df.quality_pass.sum()):,} ({df.quality_pass.mean():.1%})')

,source_id,ra,dec,ra_error,dec_error,parallax,parallax_error,pmra,pmra_error,pmdec,...,nu_eff_used_in_astrometry.mask,pseudocolour,pseudocolour.mask,tile_id,l_deg,b_deg,local_selection,parallax_corrected,parallax_snr,quality_pass
0,2057863455953995904,307.282321,37.903616,0.015252,0.017765,0.977831,0.020743,-5.862560,0.019954,-6.754822,...,False,NaN,True,tile03,77.002942,-0.585972,"l[77,83], b[-1.5,4]",0.994831,47.960326,True
1,2058051266283377024,307.234607,37.934276,0.059470,0.068418,1.012071,0.082958,-5.663563,0.080954,-9.283475,...,False,NaN,True,tile03,77.005748,-0.537506,"l[77,83], b[-1.5,4]",1.029071,12.404686,False
2,2058051472441808000,307.230651,37.947471,0.033914,0.035710,0.807556,0.046131,-8.700295,0.044139,-4.090605,...,False,NaN,True,tile03,77.014619,-0.527251,"l[77,83], b[-1.5,4]",0.824556,17.874088,True
3,2058051502500873856,307.204892,37.946760,0.080518,0.087711,1.032752,0.112401,2.745481,0.108130,11.790972,...,True,1.260523,False,tile03,77.002152,-0.511198,"l[77,83], b[-1.5,4]",1.049752,9.339348,True
4,2058052262715794048,307.195392,37.953723,0.035946,0.039473,0.702138,0.049422,-4.878425,0.048780,-2.379186,...,False,NaN,True,tile03,77.003415,-0.501049,"l[77,83], b[-1.5,4]",0.719138,14.551102,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245838,2080804319395294080,304.317016,42.667785,0.086689,0.097728,0.554649,0.109797,-6.096208,0.114788,-5.062309,...,False,NaN,True,tile02,79.617803,3.994036,"l[77,83], b[-1.5,4]",0.571649,5.206416,True
245839,2080804349450844800,304.331506,42.684422,0.102849,0.112549,0.753957,0.125203,3.121325,0.141273,-0.649794,...,False,NaN,True,tile02,79.637602,3.994463,"l[77,83], b[-1.5,4]",0.770957,6.157681,True
245840,2080804353744416000,304.326705,42.682435,0.046305,0.048533,0.433548,0.055855,-1.079875,0.061682,-1.720078,...,False,NaN,True,tile02,79.633976,3.996286,"l[77,83], b[-1.5,4]",0.450548,8.066465,True
245841,2080804486889806208,304.344083,42.701091,0.101193,0.111002,0.813992,0.125282,2.593033,0.134073,-2.169259,...,False,NaN,True,tile02,79.656640,3.996081,"l[77,83], b[-1.5,4]",0.830992,6.632993,True


## Before Section 3 — Why use density clustering and scan stability

DBSCAN searches for overdensities in (l,b,parallax,proper motion) without forcing one spherical cluster or specifying the number of groups. `eps` is a radius in RobustScaler units, not degrees; `min_samples=15` suppresses tiny chance clumps but can lose sparse outskirts. The 0.34/0.42/0.50 scan tests whether structure persists under nearby choices. The current candidate rule keeps every cluster with at least 20 stars, so it is an initial segmentation, not yet a verified Cyg OB2 selection. Noise-labelled stars may still be genuine outskirts or runaways.


In [ ]:
# Section 3 — Density clustering and stability scan
results=[]
for eps in (0.34,0.42,0.50):
    labels=DBSCAN(eps=eps,min_samples=15,n_jobs=-1).fit_predict(X)
    counts=pd.Series(labels[labels>=0]).value_counts()
    results.append({'eps':eps,'clusters':len(counts),'assigned':int((labels>=0).sum()),'largest':int(counts.max()) if len(counts) else 0})
display(pd.DataFrame(results))
labels=DBSCAN(eps=0.42,min_samples=15,n_jobs=-1).fit_predict(X)
analysis['cluster_label']=labels
sizes=analysis.loc[labels>=0].groupby('cluster_label').size()
keep=sizes[sizes>=20].index
candidate=analysis[analysis.cluster_label.isin(keep)].copy()
print('candidate rows',len(candidate))

## Before Section 4 — Why estimate membership probabilistically

A star is not simply a member or non-member: its Gaia measurement has uncertainty. We perturb parallax and proper motions and count the fraction of draws remaining within three robust scales of the candidate locus. The P>0.05 retention is a soft threshold; downstream IMF counts should use the continuous probability rather than a hard cut. The 100 draws are only a prototype (Monte Carlo noise is about 0.05 near P=0.5). The current diagonal error model ignores Gaia correlations and the global-locus test is not yet a full field-versus-cluster likelihood, so these probabilities are development values.


In [ ]:
# Section 4 — Monte Carlo membership probability
# Development approximation: diagonal reported errors. Full Gaia covariance is required for final results.
values=candidate[features].to_numpy(float); centre=np.nanmedian(values,axis=0)
scale=np.nanmedian(np.abs(values-centre),axis=0); scale=np.where(scale==0,1,scale); scale[:2]=np.maximum(scale[:2],0.05); scale[2:]=np.maximum(scale[2:],0.05)
# One uncertainty column is required for each feature: l, b, parallax, pmra, pmdec.
# Gaia sky-position errors are negligible for this first pass, so l and b use tiny floors.
err=np.column_stack([np.full((len(candidate),2),1e-5),candidate[['parallax_error','pmra_error','pmdec_error']].to_numpy(float)])
hits=np.zeros(len(candidate),dtype=int)
for _ in range(N_MC):
    draw=values+rng.normal(size=values.shape)*err
    hits += np.all(np.abs((draw-centre)/scale)<=3,axis=1)
candidate['membership_probability']=hits/N_MC
candidate['membership_status']=pd.cut(candidate.membership_probability,[-np.inf,.05,.5,.8,np.inf],labels=['rejected','tentative','probable','high'])
probable=candidate[candidate.membership_probability>.05].copy()
probable[['membership_status']].value_counts()

## Before Section 5 — Why test one versus two distance populations

Berlanas reported populations near 1.35 and 1.6 kpc. A one- versus two-component Gaussian-mixture comparison asks whether the candidate parallaxes are better described by one or two distributions; BIC penalizes the extra parameters in the two-component model. A two-component preference is not automatically proof of two physical groups: contamination, depth, binaries, measurement errors, or zero-point structure can mimic it. The labels are statistical and must later be checked against sky position, proper motion, extinction, and spectroscopy.


In [ ]:
# Section 5 — One- versus two-component corrected-parallax test
p=probable[['parallax_corrected']].to_numpy()
g1=GaussianMixture(1,random_state=SEED).fit(p); g2=GaussianMixture(2,random_state=SEED).fit(p)
distance_test=pd.DataFrame({'model':['one_component','two_component'],'bic':[g1.bic(p),g2.bic(p)]})
distance_test['delta_bic_vs_one']=distance_test.bic-distance_test.loc[0,'bic']; display(distance_test)
probable['subgroup_label']='unassigned'
if g2.bic(p)<g1.bic(p):
    raw_labels=g2.predict(p); order={old:new for new,old in enumerate(np.argsort(g2.means_.ravel()))}
    probable['subgroup_label']=[f'parallax_pop_{order[x]+1}' for x in raw_labels]
display(probable.groupby('subgroup_label').parallax_corrected.agg(['count','median','std']))

## Before Section 6 — Why we write a named catalogue and manifest

This is the WP2 handoff. We sort by `source_id` for deterministic output, retain raw and corrected astrometry plus the probability and subgroup labels, and write Parquet for efficient downstream analysis. The manifest records every threshold and approximation so later changes cannot silently alter the member list. The row-count chain (`WP1 → quality → clustered → P>0.05 → P>0.5`) distinguishes measurement-quality loss from astrophysical non-membership.


In [ ]:
# Section 6 — Write named output and audit manifest
cols=['source_id','ra','dec','l_deg','b_deg','parallax_raw','parallax_zero_point','parallax_corrected','parallax_error','pmra','pmdec','ruwe','cluster_label','membership_probability','membership_status','subgroup_label','tile_id']
members=probable[cols].sort_values('source_id').reset_index(drop=True)
members.to_parquet(OUT/'wp2_members.parquet',index=False)
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'notebook':'notebooks/wp2_membership_and_substructure.ipynb','input':str(INPUT.relative_to(ROOT)),'zero_point_recipe':'Lindegren et al. Gaia DR3/EDR3 via gaiadr3-zeropoint 0.1.0','zero_point_summary_mas':{'median':float(df.parallax_zero_point.median()),'min':float(df.parallax_zero_point.min()),'max':float(df.parallax_zero_point.max())},'quality_filter':'RUWE<1.4, visibility_periods_used>=8, BP/RP excess present','clustering':'DBSCAN eps=0.42 min_samples=15 RobustScaler','monte_carlo':{'N':N_MC,'seed':SEED,'errors':'diagonal approximation'},'counts':{'input':len(df),'analysis':len(analysis),'candidate':len(candidate),'members_p_gt_005':len(members),'members_p_gt_05':int((members.membership_probability>.5).sum())},'distance_test':distance_test.to_dict(orient='records'),'output':'data/processed/wp2_members.parquet'}
(PROV/'wp2_membership_manifest.json').write_text(json.dumps(manifest,indent=2,default=float)+'\n')
print(json.dumps(manifest['counts'],indent=2))

## Validation gate

Before calling WP2 complete, merge the Wright/Berlanas anchor lists, measure recovery at P>0.5, explain every missed anchor, enable the full astrometric covariance, and repeat with wide/control-field data. The source-dependent Lindegren zero-point recipe is now implemented; its extrapolation warnings for out-of-range magnitudes or colour proxies must be audited and included in the sensitivity analysis.